# **Convolución para MNIST-1D**

Este cuaderno explora una red convolucional 1D para MNIST-1D.

Ejecuta las celdas en orden. En varios lugares verás la palabra "TAREA": sigue esas instrucciones, anticipa qué ocurrirá o escribe el código necesario para completar las funciones.


In [ ]:
# Ejecuta esta celda una vez si necesitas instalar el repositorio MNIST-1D
!pip install git+https://github.com/greydanus/mnist1d

In [ ]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [ ]:
args = mnist1d.data.get_dataset_args()
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=False)

# Las entradas y salidas de entrenamiento y prueba están en
# data['x'], data['y'], data['x_test'] y data['y_test']
print("Samples in training set: {}".format(len(data['y'])))
print("Samples in test set: {}".format(len(data['y_test'])))
print("Length of each sample: {}".format(data['x'].shape[-1]))

In [ ]:
# Carga los datos
train_data_x = data['x'].transpose()
train_data_y = data['y']
val_data_x = data['x_test'].transpose()
val_data_y = data['y_test']
# Muestra los tamaños
print("Training data: %d samples (columns), each with %d dimensions (rows)"%((train_data_x.shape[1],train_data_x.shape[0])))
print("Validation data: %d samples (columns), each with %d dimensions (rows)"%((val_data_x.shape[1],val_data_x.shape[0])))

Define la red

In [ ]:
# Estos datos tienen 40 dimensiones de entrada y 10 dimensiones de salida
# Las entradas corresponden a los 40 desplazamientos de la plantilla MNIST-1D.
D_i = 40
# Las salidas corresponden a los 10 dígitos
D_o = 10


# TAREA: crea un modelo con las siguientes capas
# 1. Capa convolucional (entrada: longitud 40 y 1 canal; kernel 3; paso 2; padding="valid"; 15 canales de salida)
# 2. ReLU
# 3. Capa convolucional (entrada: longitud 19 y 15 canales; kernel 3; paso 2; padding="valid"; 15 canales de salida)
# 4. ReLU
# 5. Capa convolucional (entrada: longitud 9 y 15 canales; kernel 3; paso 2; padding="valid"; 15 canales de salida)
# 6. ReLU
# 7. Aplanamiento (convierte 4x15) a longitud 60
# 8. Capa lineal (tamaño de entrada = 60, tamaño de salida = 10)
# Documentación útil:
# https://pytorch.org/docs/1.13/generated/torch.nn.Conv1d.html?highlight=conv1d#torch.nn.Conv1d
# https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html
# https://pytorch.org/docs/1.13/generated/torch.nn.Linear.html?highlight=linear#torch.nn.Linear

# TEN EN CUENTA QUE LAS CAPAS CONVOLUCIONALES NECESITAN RECIBIR EL NÚMERO DE CANALES DE ENTRADA COMO PARÁMETRO
# Y NO EL TAMAÑO DE LA ENTRADA.

# Reemplaza la siguiente función:
model = nn.Sequential(

    # Primera capa convolutional
    nn.Conv1d(
        in_channels=1,
        out_channels=15,
        kernel_size=3,
        stride=2,
        padding=0
    ),
      #Se reemplazó la primera capa totalmente conectada por una convolución 1D para detectar
    #características locales de la señal MNIST-1D. Se utilizan 15 filtros de tamaño 3 con stride de 2
    #para reducir la longitud de la entrada y extraer patrones relevantes.
    nn.ReLU(),

    #Se añadió una función de activación ReLU para introducir no linealidad en el modelo
    #y permitir aprender patrones más complejos.

    # Segunda capa convolutional
    nn.Conv1d(
        in_channels=15,
        out_channels=15,
        kernel_size=3,
        stride=2,
        padding=0
    ),
        #Se agregó una segunda capa convolucional para continuar extrayendo
    #características de mayor nivel manteniendo los 15 canales de salida.

    nn.ReLU(),

      #Se mantiene la activación ReLU después de la segunda convolución para
   # conservar la capacidad de modelar relaciones no lineales.

    # Tercera capa convolutional
    nn.Conv1d(
        in_channels=15,
        out_channels=15,
        kernel_size=3,
        stride=2,
        padding=0
    ),

 #Se añadió una tercera capa convolucional que reduce nuevamente la longitud
    #de la señal y obtiene representaciones más compactas antes de la clasificación.

    nn.ReLU(),
#Se aplica otra activación ReLU para mejorar la capacidad de aprendizaje de la red.

    # Convierte (15,4) en un vector de 60 elementos
    nn.Flatten(),
    #Se aplana la salida de las convoluciones (15 canales × 4 posiciones = 60 valores)
   # para poder conectarla a la capa totalmente conectada final.

    # Clasificación final
    nn.Linear(60, 10)
)

#La última capa lineal transforma los 60 atributos obtenidos por la red convolucional en
#las 10 clases correspondientes a los dígitos del conjunto MNIST-1D.

# esto es una Fully conected, tienes que cambiarla a convolutional
# usa nn.Conv1d y nn.ReLU para las capas de convolucionales y nn.Flatten para aplanar la salida antes de la capa lineal final





In [ ]:
# Inicialización de pesos de He
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

In [ ]:
# Elige la función de pérdida de entropía cruzada
loss_function = nn.CrossEntropyLoss()
# Construye el optimizador SGD e inicializa la tasa de aprendizaje y el momentum
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05, momentum=0.9)
# Objeto que reduce la tasa de aprendizaje a la mitad cada 20 épocas
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)
# Crea los tensores de datos y guárdalos en un DataLoader
x_train = torch.tensor(train_data_x.transpose().astype('float32'))
y_train = torch.tensor(train_data_y.astype('long')).long()
x_val= torch.tensor(val_data_x.transpose().astype('float32'))
y_val = torch.tensor(val_data_y.astype('long')).long()

# Carga los datos en una clase que crea los lotes
data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=100, shuffle=True, worker_init_fn=np.random.seed(1))

# Inicializa los pesos del modelo
model.apply(weights_init)

# Recorre el conjunto de datos n_epoch veces
n_epoch = 100
# Guarda la pérdida y el porcentaje de error en cada época
losses_train = np.zeros((n_epoch))
errors_train = np.zeros((n_epoch))
losses_val = np.zeros((n_epoch))
errors_val = np.zeros((n_epoch))

for epoch in range(n_epoch):
  # Recorre los lotes
  for i, data in enumerate(data_loader):
    # Recupera entradas y etiquetas de este lote
    x_batch, y_batch = data
    # Reinicia los gradientes de los parámetros
    optimizer.zero_grad()
    # Paso hacia adelante: calcula la salida del modelo
    pred = model(x_batch[:,None,:])
    # Calcula la pérdida
    loss = loss_function(pred, y_batch)
    # Paso hacia atrás
    loss.backward()
    # Actualización SGD
    optimizer.step()

  # Ejecuta todo el conjunto de datos para obtener estadísticas; normalmente no harías esto
  pred_train = model(x_train[:,None,:])
  pred_val = model(x_val[:,None,:])
  _, predicted_train_class = torch.max(pred_train.data, 1)
  _, predicted_val_class = torch.max(pred_val.data, 1)
  errors_train[epoch] = 100 - 100 * (predicted_train_class == y_train).float().sum() / len(y_train)
  errors_val[epoch]= 100 - 100 * (predicted_val_class == y_val).float().sum() / len(y_val)
  losses_train[epoch] = loss_function(pred_train, y_train).item()
  losses_val[epoch]= loss_function(pred_val, y_val).item()
  print(f'Epoch {epoch:5d}, training loss {losses_train[epoch]:.6f}, training error {errors_train[epoch]:3.2f}, validation loss {losses_val[epoch]:.6f}, validation error {errors_val[epoch]:3.2f}')

  # Indica al programador que evalúe si debe actualizar la tasa de aprendizaje
  scheduler.step()

# Grafica los resultados
fig, ax = plt.subplots()
ax.plot(errors_train,'r-',label='entrenamiento')
ax.plot(errors_val,'b-',label='validación')
ax.set_ylim(0,100); ax.set_xlim(0,n_epoch)
ax.set_xlabel('Epoch'); ax.set_ylabel('Error')
ax.legend()
plt.show()